# Module 7.1: Benchmarking & Cost Analysis

## Learning Objectives

By completing this notebook, you will:
1. Build an evaluation framework for trained agents
2. Calculate training and inference costs
3. Compare RL-trained models against frontier APIs
4. Make data-driven decisions about when to train custom models

## Estimated Time: 15 minutes

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. ART-E Benchmark Results

OpenPipe benchmarked their ART-trained agent (ART-E) on email search accuracy.
A 14B open-source model trained with RL outperformed frontier models.

In [ ]:
# Real benchmark data from the ART-E paper
benchmarks = {
    "ART-E (Qwen2.5-14B)": {"accuracy": 96, "latency_s": 1.1, "cost_per_1k": 0.85},
    "o3": {"accuracy": 91, "latency_s": 5.6, "cost_per_1k": 55.19},
    "o4-mini": {"accuracy": 88, "latency_s": 3.2, "cost_per_1k": 12.50},
    "Gemini 2.5 Pro": {"accuracy": 85, "latency_s": 4.1, "cost_per_1k": 18.00},
    "GPT-4.1": {"accuracy": 82, "latency_s": 2.8, "cost_per_1k": 25.00},
    "Base Qwen2.5-14B": {"accuracy": 40, "latency_s": 1.0, "cost_per_1k": 0.80},
}

models = list(benchmarks.keys())
accuracies = [b["accuracy"] for b in benchmarks.values()]
latencies = [b["latency_s"] for b in benchmarks.values()]
costs = [b["cost_per_1k"] for b in benchmarks.values()]

# Accuracy comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ["green" if m.startswith("ART") else "steelblue" if "Base" not in m else "gray" for m in models]

axes[0].barh(models, accuracies, color=colors)
axes[0].set_xlabel("Accuracy (%)")
axes[0].set_title("Email Search Accuracy")
axes[0].set_xlim(0, 100)

axes[1].barh(models, latencies, color=colors)
axes[1].set_xlabel("Latency (seconds)")
axes[1].set_title("Query Latency")

axes[2].barh(models, costs, color=colors)
axes[2].set_xlabel("Cost per 1,000 runs ($)")
axes[2].set_title("Inference Cost")

plt.suptitle("ART-E vs Frontier Models", fontsize=14)
plt.tight_layout()
plt.show()

print("Key findings:")
print(f"  ART-E accuracy: {benchmarks['ART-E (Qwen2.5-14B)']['accuracy']}% (+56% over base)")
print(f"  5x faster than o3 ({benchmarks['ART-E (Qwen2.5-14B)']['latency_s']}s vs {benchmarks['o3']['latency_s']}s)")
print(f"  64x cheaper ($0.85 vs $55.19 per 1,000 runs)")

## 2. Training Cost Calculator

How much does it cost to train your own agent?

In [ ]:
def calculate_training_cost(
    model_size_b: float,
    num_steps: int,
    group_size: int,
    num_scenarios: int,
    gpu_type: str = "A100_80GB",
) -> dict:
    """
    Calculate total training cost.
    
    Returns breakdown of GPU and judge costs.
    """
    # GPU pricing (spot instance rates)
    gpu_rates = {
        "A100_40GB": 1.50,
        "A100_80GB": 2.00,
        "H100_80GB": 3.50,
        "RTX_4090": 0.75,
    }
    rate = gpu_rates.get(gpu_type, 2.00)
    
    # GPU hours (empirical from ART benchmarks)
    # ~0.04 GPU-hours per step per B parameters for group_size=4
    hours_per_step = 0.04 * (model_size_b / 3) * (group_size / 4)
    total_gpu_hours = hours_per_step * num_steps
    gpu_cost = total_gpu_hours * rate
    
    # Judge LLM cost (RULER)
    # ~1000 tokens per trajectory, group_size trajectories per scenario per step
    tokens_per_step = num_scenarios * group_size * 1000
    total_tokens = tokens_per_step * num_steps
    judge_cost = (total_tokens / 1_000_000) * 3.0  # ~$3/M for o4-mini
    
    total = gpu_cost + judge_cost
    
    return {
        "gpu_hours": round(total_gpu_hours, 1),
        "gpu_cost": round(gpu_cost, 2),
        "judge_tokens_m": round(total_tokens / 1_000_000, 1),
        "judge_cost": round(judge_cost, 2),
        "total_cost": round(total, 2),
        "gpu_type": gpu_type,
    }


# Compare different setups
configs = [
    ("Small (3B, 50 steps)", 3, 50, 4, 20),
    ("Medium (7B, 100 steps)", 7, 100, 4, 20),
    ("Large (14B, 200 steps)", 14, 200, 8, 30),
]

print(f"{'Config':<25} {'GPU Hours':>10} {'GPU Cost':>10} {'Judge Cost':>12} {'Total':>10}")
print("-" * 70)
for name, size, steps, gs, scenarios in configs:
    cost = calculate_training_cost(size, steps, gs, scenarios)
    print(f"{name:<25} {cost['gpu_hours']:>9.1f}h ${cost['gpu_cost']:>8.2f} ${cost['judge_cost']:>10.2f} ${cost['total_cost']:>8.2f}")

## 3. Break-Even Analysis

When does training a custom model pay for itself vs using frontier APIs?

In [ ]:
def break_even_analysis(
    training_cost: float,
    custom_cost_per_query: float,
    api_cost_per_query: float,
    max_queries: int = 100_000,
) -> dict:
    """
    Calculate when custom model becomes cheaper than API.
    """
    if custom_cost_per_query >= api_cost_per_query:
        return {"break_even_queries": float('inf'), "message": "Custom never breaks even"}
    
    savings_per_query = api_cost_per_query - custom_cost_per_query
    break_even = training_cost / savings_per_query
    
    return {
        "break_even_queries": int(break_even),
        "savings_at_100k": round((savings_per_query * 100_000) - training_cost, 2),
    }


# Custom 14B model vs o3 API
training_cost = 80  # $80 for 14B model training
custom_per_query = 0.85 / 1000  # $0.00085 per query
o3_per_query = 55.19 / 1000     # $0.05519 per query

result = break_even_analysis(training_cost, custom_per_query, o3_per_query)

print(f"Training cost: ${training_cost}")
print(f"Custom model: ${custom_per_query:.5f}/query")
print(f"o3 API: ${o3_per_query:.5f}/query")
print(f"Break-even at: {result['break_even_queries']:,} queries")
print(f"Savings at 100k queries: ${result['savings_at_100k']:,.2f}")

# Visualize
queries = np.arange(0, 5001)
api_total = queries * o3_per_query
custom_total = training_cost + queries * custom_per_query

plt.figure(figsize=(10, 5))
plt.plot(queries, api_total, label="o3 API", color="red", linewidth=2)
plt.plot(queries, custom_total, label="ART-trained 14B", color="green", linewidth=2)
plt.axvline(result['break_even_queries'], color="gray", linestyle="--", alpha=0.5)
plt.text(result['break_even_queries'] + 50, training_cost + 20,
         f"Break-even\n{result['break_even_queries']:,} queries", fontsize=10)
plt.xlabel("Number of Queries")
plt.ylabel("Total Cost ($)")
plt.title("Cost Comparison: Custom RL Model vs Frontier API")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Evaluation Framework

How to systematically evaluate your trained agent.

In [ ]:
from dataclasses import dataclass


@dataclass
class EvalResult:
    scenario: str
    expected: str
    actual: str
    correct: bool
    num_turns: int
    latency_s: float
    had_errors: bool


def evaluate_agent(
    eval_scenarios: list[dict],
    run_fn,
) -> dict:
    """
    Evaluate an agent on held-out scenarios.
    
    Parameters
    ----------
    eval_scenarios : list[dict]
        Each has 'question' and 'expected_answer' keys
    run_fn : callable
        Function that takes a question and returns (answer, num_turns, latency, had_errors)
    
    Returns
    -------
    dict with accuracy, avg_turns, avg_latency, error_rate
    """
    results = []
    for scenario in eval_scenarios:
        answer, turns, latency, errors = run_fn(scenario["question"])
        results.append(EvalResult(
            scenario=scenario["question"],
            expected=scenario["expected_answer"],
            actual=answer,
            correct=(answer == scenario["expected_answer"]),
            num_turns=turns,
            latency_s=latency,
            had_errors=errors,
        ))
    
    correct = sum(1 for r in results if r.correct)
    return {
        "accuracy": correct / len(results),
        "avg_turns": np.mean([r.num_turns for r in results]),
        "avg_latency": np.mean([r.latency_s for r in results]),
        "error_rate": np.mean([r.had_errors for r in results]),
        "results": results,
    }


# Simulated evaluation
eval_scenarios = [
    {"question": "Highest-paid employee", "expected_answer": "Bob Martinez"},
    {"question": "Count of active projects", "expected_answer": "5"},
    {"question": "Engineering dept budget", "expected_answer": "2500000"},
    {"question": "Newest hire", "expected_answer": "Frank Lee"},
    {"question": "Total project budgets", "expected_answer": "2430000"},
]


def sim_base_model(question):
    """Simulated base model (40% accuracy)."""
    correct = np.random.random() < 0.4
    answer = eval_scenarios[0]["expected_answer"] if correct else "unknown"
    return answer, np.random.randint(1, 6), np.random.uniform(0.5, 2.0), np.random.random() < 0.4


def sim_trained_model(question):
    """Simulated trained model (90% accuracy)."""
    correct = np.random.random() < 0.9
    scenario = next((s for s in eval_scenarios if s["question"] == question), eval_scenarios[0])
    answer = scenario["expected_answer"] if correct else "unknown"
    return answer, np.random.randint(2, 4), np.random.uniform(0.8, 1.5), np.random.random() < 0.1


base_eval = evaluate_agent(eval_scenarios, sim_base_model)
trained_eval = evaluate_agent(eval_scenarios, sim_trained_model)

print(f"{'Metric':<20} {'Base Model':>12} {'Trained':>12}")
print("-" * 45)
print(f"{'Accuracy':<20} {base_eval['accuracy']:>11.0%} {trained_eval['accuracy']:>11.0%}")
print(f"{'Avg Turns':<20} {base_eval['avg_turns']:>12.1f} {trained_eval['avg_turns']:>12.1f}")
print(f"{'Avg Latency':<20} {base_eval['avg_latency']:>11.2f}s {trained_eval['avg_latency']:>11.2f}s")
print(f"{'Error Rate':<20} {base_eval['error_rate']:>11.0%} {trained_eval['error_rate']:>11.0%}")

## Exercise: Decision Framework

**Task:** Given a real scenario, calculate whether training a custom model is worth it.

Scenario: You have an agent making 50,000 queries/month.
- Current: Using GPT-4.1 at $0.025 per query
- Option: Train a Qwen2.5-7B with ART for $45 training cost, $0.001 per query inference

In [ ]:
# YOUR CODE HERE
# ---------------

monthly_queries = 50_000
months = 6

# TODO: Calculate total costs for each option over 6 months
api_total_cost = 0.0       # GPT-4.1 cost for 6 months
custom_total_cost = 0.0    # Training + inference for 6 months
savings = 0.0              # How much you save with custom model

exercise_api_cost = api_total_cost
exercise_custom_cost = custom_total_cost
exercise_savings = savings

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------

def test_exercise():
    expected_api = 50_000 * 0.025 * 6  # $7,500
    expected_custom = 45 + (50_000 * 0.001 * 6)  # $345
    expected_savings = expected_api - expected_custom
    
    assert abs(exercise_api_cost - expected_api) < 1, \
        f"API cost should be ~${expected_api}, got ${exercise_api_cost}"
    assert abs(exercise_custom_cost - expected_custom) < 1, \
        f"Custom cost should be ~${expected_custom}, got ${exercise_custom_cost}"
    assert exercise_savings > 0, "Custom model should save money"
    assert abs(exercise_savings - expected_savings) < 1, \
        f"Savings should be ~${expected_savings}, got ${exercise_savings}"
    
    print(f"All tests passed!")
    print(f"  API cost (6 months): ${exercise_api_cost:,.2f}")
    print(f"  Custom cost (6 months): ${exercise_custom_cost:,.2f}")
    print(f"  Savings: ${exercise_savings:,.2f} ({exercise_savings/exercise_api_cost:.0%})")

test_exercise()

## Summary

**Key Takeaways:**
1. ART-trained 14B model beat o3 at 96% accuracy, 5x faster, 64x cheaper
2. Training costs are modest: $10-80 depending on model size
3. Break-even happens quickly at scale (often < 2,000 queries)
4. Systematic evaluation requires held-out scenarios and multiple metrics

**Course Complete!** You now have the full toolkit to train agents with RL.